# Gemma 4 Batch Evaluation

This notebook evaluates the Google Gemma 4 model on a dataset of audio segments.
It uses the direct batch inference approach with AutoModelForMultimodalLM.

In [ ]:
# @title Install packages
import builtins


# The Magic Hack: Create a dummy class and inject it into Python's builtins
# so the Python 3.12 type-hint evaluator finds it and stops crashing.
class DummyPeftConfig:
    pass


builtins.PeftConfigLike = DummyPeftConfig

import os
import json
import sys
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM
from google.cloud import storage
from huggingface_hub import login

# Import common utils
from common.gcs_utils import download_jsonl_manifest, upload_inference_results
from common.inference_pipeline_runner import run_inference_pipeline

# Configure logging
import logging

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

print("Imports finally successful!")

In [ ]:
# --- Configuration ---
MODEL_NAME = "google/gemma-4-E2B-it"
SELECTED_MODEL_KEY = "gemma-4-E2B-it"

# @markdown ### Google Cloud Platform Settings
GCP_PROJECT_ID = "<YOUR_GCP_PROJECT_ID>"  # @param {type:"string"}
GCS_MANIFEST_URI = "<YOUR_GCS_MANIFEST_URI>"  # @param {type:"string"}
GCS_BUCKET = "<YOUR_GCS_BUCKET_NAME>"  # @param {type:"string"}

# @markdown ### Evaluation Tracking
PROJECT_NAME = "<YOUR_PROJECT_NAME>"  # @param {type:"string"}
EXPERIMENT_NAME = "<YOUR_EXPERIMENT_NAME>"  # @param {type:"string"}

# @markdown ### Inference Parameters
BATCH_SIZE = 4  # @param {type:"integer"}
LIMIT = 10  # @param {type:"integer"}

# Log in to Hugging Face.
# This expects 'HF_TOKEN' to be set in Google Colab Secrets or as an environment variable.
# If neither is found, it will securely prompt you for the token.
from common.auth_utils import login_to_huggingface

login_to_huggingface()

In [ ]:
# @title Load the model
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading Gemma model on {device}...")

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_NAME, torch_dtype="auto", device_map="auto"
)
processor = AutoProcessor.from_pretrained(MODEL_NAME)

In [ ]:
# @title Define helper functions for evaluation runner

from common.audio_utils import preprocess_audio_for_model
import os
import torch


def prompt_formatter(entry, local_path):
    """Constructs the message structure for Gemma 4 as per Google Doc."""
    return [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Transcribe the following speech segment in its original language. Follow these specific instructions for formatting the answer:\
* Only output the transcription, with no newlines.\
* When transcribing numbers, write the digits, i.e. write 1.7 and not one point seven, and write 3 instead of three.",
                },
                {"type": "audio", "audio": local_path},
            ],
        }
    ]


def find_audio_item(prompt):
    """Helper to find the audio dictionary item in the prompt structure."""
    for message in prompt:
        content = message.get("content", [])
        if isinstance(content, list):
            for item in content:
                if isinstance(item, dict) and item.get("type") == "audio":
                    return item
    return None


def gemma_inference(model, prompts):
    """Runs inference using processor.apply_chat_template directly as in Google Doc."""
    outputs = []
    for p in prompts:
        audio_item = find_audio_item(p)
        if not audio_item:
            continue

        audio_path = audio_item["audio"]
        # audio_path is already the preprocessed WAV file path!

        try:
            # Prepare inputs using processor
            input_ids = processor.apply_chat_template(
                p,
                add_generation_prompt=True,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
            )
            input_ids = input_ids.to(model.device, dtype=model.dtype)

            # Generate
            out = model.generate(
                **input_ids, max_new_tokens=128, do_sample=False
            )

            # Decode only the new tokens (slicing off the prompt)
            transcription = processor.batch_decode(
                out[:, input_ids["input_ids"].shape[1] :],
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False,
            )[0]

            outputs.append(transcription)

        except Exception as e:
            logger.error(f"Failed during inference for {audio_path}: {e}")
            outputs.append("[ERROR]")

    return outputs


def result_decoder(ans, model):
    """Since we decoded in gemma_inference, this just returns the string."""
    return ans.strip()

In [ ]:
# @title Run Evaluation

storage_client = storage.Client(project=GCP_PROJECT_ID)
manifest_data = download_jsonl_manifest(storage_client, GCS_MANIFEST_URI)

# Run the generic batch evaluation
results_list = run_inference_pipeline(
    model=model,
    manifest_data=manifest_data,
    prompt_fn=prompt_formatter,
    inference_fn=gemma_inference,
    decode_fn=result_decoder,
    preprocess_fn=preprocess_audio_for_model,
    storage_client=storage_client,
    project_name=PROJECT_NAME,
    selected_model=SELECTED_MODEL_KEY,
    batch_size=BATCH_SIZE,
    limit=LIMIT,
)

In [ ]:
# Upload results directly to GCS from memory (uncomment to use)
gcs_uri = upload_inference_results(
    storage_client,
    GCS_BUCKET,
    PROJECT_NAME,
    SELECTED_MODEL_KEY,
    EXPERIMENT_NAME,
    results_list,
)